<a href="https://colab.research.google.com/github/The-cheater/Deep_Learning_Models/blob/main/final_correct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================
# 0️⃣ Connect Google Drive & Setup
# ==========================
from google.colab import drive
import zipfile
import os

# Mount Google Drive
drive.mount('/content/drive')

# Unzip GAF Images
with zipfile.ZipFile('/content/drive/MyDrive/dataset/GAF_Images.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/GAF_Images')

# Unzip MTF Images
with zipfile.ZipFile('/content/drive/MyDrive/dataset/MTF_Images.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/MTF_Images')

# Check extracted folders
print("GAF Images:", os.listdir('/content/GAF_Images')[:5])
print("MTF Images:", os.listdir('/content/MTF_Images')[:5])


Mounted at /content/drive
GAF Images: ['GAF_Images_train', 'GAF_Images_test']
MTF Images: ['MTF_Images_train', 'MTF_Images_test']


In [2]:
# ==========================
# 1️⃣ Install Required Packages
# ==========================
!pip install --upgrade pip
!pip install tensorflow opencv-python matplotlib

# ==========================
# 2️⃣ Imports & Config
# ==========================
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping

label_map = {'EL': 0, 'PD': 1, 'S': 2}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [3]:
# ==========================
# 3️⃣ Data Loader Functions
# ==========================
def parse_pair(gaf_path, mtf_path, label):
    gaf_img = tf.io.read_file(gaf_path)
    gaf_img = tf.image.decode_png(gaf_img, channels=3)
    gaf_img = tf.image.resize(gaf_img, [224, 224])
    gaf_img = tf.cast(gaf_img, tf.float32) / 255.0

    mtf_img = tf.io.read_file(mtf_path)
    mtf_img = tf.image.decode_png(mtf_img, channels=3)
    mtf_img = tf.image.resize(mtf_img, [224, 224])
    mtf_img = tf.cast(mtf_img, tf.float32) / 255.0

    return {'gaf_input': gaf_img, 'mtf_input': mtf_img}, label

def create_file_paths_and_labels(gaf_root, mtf_root):
    gaf_paths, mtf_paths, labels = [], [], []

    for root, _, files in os.walk(gaf_root):
        for file in files:
            if file.lower().endswith('_gaf.png'):
                gaf_path = os.path.join(root, file)
                mtf_path = gaf_path.replace('GAF_Images', 'MTF_Images').replace('_gaf.png', '_mtf.png')
                if not os.path.exists(mtf_path):
                    continue

                parts = gaf_path.split(os.sep)
                for cn in label_map:
                    if cn in parts:
                        labels.append(label_map[cn])
                        gaf_paths.append(gaf_path)
                        mtf_paths.append(mtf_path)
                        break

    return gaf_paths, mtf_paths, labels

# ==========================
# 4️⃣ Dataset Pipeline
# ==========================
gaf_root = '/content/GAF_Images/GAF_Images_train'
mtf_root = '/content/MTF_Images/MTF_Images_train'

gaf_paths, mtf_paths, labels = create_file_paths_and_labels(gaf_root, mtf_root)
print(f"✅ Found {len(gaf_paths)} samples.")

gaf_train, gaf_val, mtf_train, mtf_val, labels_train, labels_val = train_test_split(
    gaf_paths, mtf_paths, labels, test_size=0.2, random_state=42, stratify=labels)

print(f"✅ Training samples: {len(gaf_train)}, Validation samples: {len(gaf_val)}")

batch_size = 16

def create_dataset(gaf, mtf, labels):
    ds = tf.data.Dataset.from_tensor_slices((gaf, mtf, labels))
    ds = ds.map(parse_pair, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_dataset = create_dataset(gaf_train, mtf_train, labels_train).shuffle(10000)
val_dataset = create_dataset(gaf_val, mtf_val, labels_val)


✅ Found 8034 samples.
✅ Training samples: 6427, Validation samples: 1607


In [4]:
# ==========================
# 5️⃣ Model Definition
# ==========================
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)
    return x

def channel_attention_module(input_feature, ratio=8):
    channel = input_feature.shape[-1]
    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)
    shared_dense_one = layers.Dense(channel // ratio, activation='relu')
    shared_dense_two = layers.Dense(channel)
    avg_out = shared_dense_two(shared_dense_one(avg_pool))
    max_out = shared_dense_two(shared_dense_one(max_pool))
    cbam_feature = layers.Add()([avg_out, max_out])
    cbam_feature = layers.Activation('sigmoid')(cbam_feature)
    cbam_feature = layers.Reshape((1, 1, channel))(cbam_feature)
    return layers.Multiply()([input_feature, cbam_feature])

input_gaf = Input(shape=(224, 224, 3), name='gaf_input')
input_mtf = Input(shape=(224, 224, 3), name='mtf_input')

x1 = conv_block(input_gaf, 64)
x1 = conv_block(x1, 128)
x1 = conv_block(x1, 256)
x1 = conv_block(x1, 256)

x2 = conv_block(input_mtf, 64)
x2 = conv_block(x2, 128)
x2 = conv_block(x2, 256)
x2 = conv_block(x2, 256)

merged = layers.Concatenate(axis=-1)([x1, x2])
x = conv_block(merged, 512)
x = layers.Conv2D(512, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = channel_attention_module(x)
x = layers.Flatten()(x)
output = layers.Dense(3, activation='softmax')(x)

model = Model(inputs=[input_gaf, input_mtf], outputs=output)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("\n📐 Model Summary:")
model.summary()


📐 Model Summary:


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ gaf_input           │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mtf_input           │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 224, 224,  │      1,792 │ gaf_input[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 224, 224,  │      1,792 │ mtf_input[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 224, 224,  │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 224, 224,  │        256 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 112, 112,  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 112, 112,  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 112, 112,  │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 112, 112,  │     73,856 │ max_pooling2d_4[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        512 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        512 │ conv2d_5[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 56, 56,    │    295,168 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 56, 56,    │    295,168 │ max_pooling2d_5[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │      1,024 │ conv2d_2[0][0]  

 Total params: 6,792,515 (25.91 MB)

 Trainable params: 6,787,651 (25.89 MB)

 Non-trainable params: 4,864 (19.00 KB)

In [5]:
# ==========================
# 6️⃣ Training
# ==========================
early_stop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

print("\n🏋️‍♂️ Starting model training...")
history = model.fit(
    train_dataset,
    epochs=20,
    validation_data=val_dataset,
    callbacks=[early_stop]
)
print("✅ Training complete.")


🏋️‍♂️ Starting model training...
Epoch 1/20
402/402 ━━━━━━━━━━━━━━━━━━━━ 144s 199ms/step - accuracy: 0.5326 - loss: 1.3313 - val_accuracy: 0.2651 - val_loss: 2.5997
Epoch 2/20
402/402 ━━━━━━━━━━━━━━━━━━━━ 118s 169ms/step - accuracy: 0.6071 - loss: 0.8051 - val_accuracy: 0.6783 - val_loss: 0.7174
Epoch 3/20
402/402 ━━━━━━━━━━━━━━━━━━━━ 93s 169ms/step - accuracy: 0.7267 - loss: 0.6173 - val_accuracy: 0.7424 - val_loss: 0.6113
Epoch 4/20
402/402 ━━━━━━━━━━━━━━━━━━━━ 92s 164ms/step - accuracy: 0.8325 - loss: 0.3963 - val_accuracy: 0.8749 - val_loss: 0.3210
Epoch 5/20
402/402 ━━━━━━━━━━━━━━━━━━━━ 92s 161ms/step - accuracy: 0.9319 - loss: 0.1806 - val_accuracy: 0.8793 - val_loss: 0.3095
Epoch 6/20
402/402 ━━━━━━━━━━━━━━━━━━━━ 136s 157ms/step - accuracy: 0.9671 - loss: 0.0913 - val_accuracy: 0.9452 - val_loss: 0.1597
Epoch 7/20
402/402 ━━━━━━━━━━━━━━━━━━━━ 150s 168ms/step - accuracy: 0.9848 - loss: 0.0434 - val_accuracy: 0.9359 - val_loss: 0.1784
Epoch 8/20
402/402 ━━━━━━━━━━━━━━━━━━━━ 140s 

In [6]:
model.save('/content/drive/MyDrive/17_06.h5')


In [7]:
from tensorflow import keras

# Load the saved model
model = keras.models.load_model('/content/drive/MyDrive/17_06.h5')


In [15]:
import os
import numpy as np
from tensorflow import keras
import tensorflow as tf
from sklearn.metrics import confusion_matrix, classification_report

# ————————————————————————————————
# 🔄 1. Load your trained model from Drive
# ————————————————————————————————
model_path = '/content/drive/MyDrive/17_06.h5'
model = keras.models.load_model(model_path)
print(f"✅ Model loaded from: {model_path}")

# ————————————————————————————————
# 📂 2. Define or load your test dataset
# Replace the paths and logic below to match your setup
# You need two parallel lists: test_gaf_paths, test_mtf_paths, and test_labels
# ————————————————————————————————
# Example loader (modify roots & label-mapping accordingly)
def parse_pair(gaf_path, mtf_path, label):
    gaf = tf.io.read_file(gaf_path)
    gaf = tf.image.decode_png(gaf, channels=3)
    gaf = tf.image.resize(gaf, [224, 224]) / 255.0

    mtf = tf.io.read_file(mtf_path)
    mtf = tf.image.decode_png(mtf, channels=3)
    mtf = tf.image.resize(mtf, [224, 224]) / 255.0

    # The functional model expects inputs as a tuple or list
    return (gaf, mtf), label

# Example: define your dataset paths & labels
TEST_GAF_ROOT = '/content/GAF_Images/GAF_Images_test'
# FIX: Correct the path for the MTF test images
TEST_MTF_ROOT = '/content/MTF_Images/MTF_Images_test'
label_map = {'EL':0, 'PD':1, 'S':2}

def make_test_lists(gaf_root, mtf_root):
    gaf_paths, mtf_paths, labels = [], [], []
    for root, _, files in os.walk(gaf_root):
        for f in files:
            if f.endswith('_gaf.png'):
                gaf_p = os.path.join(root, f)
                # Ensure the replacement correctly points to the MTF root
                mtf_p = gaf_p.replace(gaf_root, mtf_root).replace('_gaf.png','_mtf.png')
                if os.path.exists(mtf_p):
                    for cn, idx in label_map.items():
                        if cn in root or cn in f:
                            labels.append(idx)
                            gaf_paths.append(gaf_p)
                            mtf_paths.append(mtf_p)
                            break
    return gaf_paths, mtf_paths, labels

test_gaf_paths, test_mtf_paths, test_labels = make_test_lists(TEST_GAF_ROOT, TEST_MTF_ROOT)
print(f"🔍 Found {len(test_labels)} test samples.")

# Add print statements to verify the contents of the lists
print(f"Sample test_gaf_paths (first 5): {test_gaf_paths[:5]}")
print(f"Sample test_mtf_paths (first 5): {test_mtf_paths[:5]}")
print(f"Sample test_labels (first 5): {test_labels[:5]}")


# ————————————————————————————————
# 3. Create a tf.data dataset for testing
# ————————————————————————————————
batch_size = 16
test_ds = tf.data.Dataset.from_tensor_slices((test_gaf_paths, test_mtf_paths, test_labels))
test_ds = test_ds.map(lambda g, m, l: parse_pair(g, m, l), tf.data.AUTOTUNE)
test_ds = test_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
print(f"📊 Test dataset prepared with batch size = {batch_size}")

# ————————————————————————————————
# 4. Run evaluation
# ————————————————————————————————
print("🏁 Starting model evaluation on test data...")
y_true, y_pred = [], []

# Check if the dataset is empty before iterating
# Compare the result of the function call with the special value
# FIX: Correctly check if the dataset cardinality is unknown or infinite
if tf.data.experimental.cardinality(test_ds).numpy() < 0: # A cardinality < 0 indicates unknown or infinite
    print("⚠️ Test dataset is empty or of unknown size. Cannot perform evaluation.")
else:
    for (g_batch, m_batch), l_batch in test_ds:
        # Ensure inputs are correctly structured as a list for the functional model
        preds = model.predict([g_batch, m_batch], verbose=0)
        y_true.extend(l_batch.numpy())
        y_pred.extend(np.argmax(preds, axis=1))

    print("✅ Evaluation completed.")

    # ————————————————————————————————
    # 5. Print detailed metrics
    # ————————————————————————————————
    print("\n📋 Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    print("\n📈 Classification Report:")
    target_names = [cn for cn, _ in sorted(label_map.items(), key=lambda x: x[1])]
    print(classification_report(y_true, y_pred, target_names=target_names))

    accuracy = np.mean(np.array(y_true) == np.array(y_pred)) * 100
    print(f"\n🎯 Overall Test Accuracy: {accuracy:.2f}%")

    # ————————————————————————————————
    # 6. Sample per-batch prediction details
    # ————————————————————————————————
    print("\n🧩 Sample predictions from the first batch:")
    for i, ((g_img, m_img), true_lbl) in enumerate(test_ds.take(1).unbatch().batch(batch_size)):
        # Ensure inputs are correctly structured as a list for the functional model
        preds = model.predict([g_img, m_img], verbose=0)
        pred_labels = np.argmax(preds, axis=1)
        for j in range(len(pred_labels)):
            print(f"  • Sample {j+1} — True: {target_names[true_lbl.numpy()[j]]}, Predicted: {target_names[pred_labels[j]]}")
        break

✅ Model loaded from: /content/drive/MyDrive/17_06.h5
🔍 Found 1030 test samples.
Sample test_gaf_paths (first 5): ['/content/GAF_Images/GAF_Images_test/EL/EL001_260220/el001_2tug2/channel_40_gaf.png', '/content/GAF_Images/GAF_Images_test/EL/EL001_260220/el001_2tug2/channel_44_gaf.png', '/content/GAF_Images/GAF_Images_test/EL/EL001_260220/el001_2tug2/channel_43_gaf.png', '/content/GAF_Images/GAF_Images_test/EL/EL001_260220/el001_2tug2/channel_42_gaf.png', '/content/GAF_Images/GAF_Images_test/EL/EL001_260220/el001_2tug2/channel_41_gaf.png']
Sample test_mtf_paths (first 5): ['/content/MTF_Images/MTF_Images_test/EL/EL001_260220/el001_2tug2/channel_40_mtf.png', '/content/MTF_Images/MTF_Images_test/EL/EL001_260220/el001_2tug2/channel_44_mtf.png', '/content/MTF_Images/MTF_Images_test/EL/EL001_260220/el001_2tug2/channel_43_mtf.png', '/content/MTF_Images/MTF_Images_test/EL/EL001_260220/el001_2tug2/channel_42_mtf.png', '/content/MTF_Images/MTF_Images_test/EL/EL001_260220/el001_2tug2/channel_41_mt

In [16]:
# prompt: print flops and gflops for this model

import tensorflow as tf
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

# Define the function to calculate FLOPs
def get_flops(model):
    # Create dummy input tensors with the expected shapes
    # These shapes should match the input shapes of your model
    dummy_gaf_input = tf.TensorSpec(shape=(1, 224, 224, 3), dtype=tf.float32)
    dummy_mtf_input = tf.TensorSpec(shape=(1, 224, 224, 3), dtype=tf.float32)

    # Trace the model's call method to get a ConcreteFunction
    # For functional models with multiple inputs, pass a list or tuple of specifications
    concrete_func = tf.function(model).get_concrete_function([dummy_gaf_input, dummy_mtf_input])
    concrete_func = convert_variables_to_constants_v2(concrete_func)

    frozen_func = concrete_func.graph.as_graph_def()

    with tf.Graph().as_default() as graph:
        tf.graph_util.import_graph_def(frozen_func, name='')

    run_meta = tf.compat.v1.RunMetadata()
    opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
    flops = tf.compat.v1.profiler.profile(graph, run_meta=run_meta, options=opts).total_float_ops

    return flops

# Assuming 'model' is already defined and is your trained TensorFlow functional model
# If 'model' is not defined, load it first:
# from tensorflow import keras
# model = keras.models.load_model('/content/drive/MyDrive/17_06.h5')


# Calculate and print FLOPs and GFLOPs
flops = get_flops(model)
print(f"\n🧠 Model FLOPs: {flops}")
print(f"🧠 Model GFLOPs: {flops / 1e9}")


Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.



🧠 Model FLOPs: 10798830994
🧠 Model GFLOPs: 10.798830994
